# Exploring the TCGA pan-cancer RNA-seq data

Before training anything, this notebook looks at the data itself: how many samples each cancer type has, what the expression values look like, and whether the tumor types already separate in an unsupervised view (PCA and t-SNE). If they do, we should expect a classifier to do very well, and the interesting question becomes *which* genes drive the separation.

Run `python -m src.download_data` from the project root first.

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

from src.data import load_data, summarize

sns.set_theme(style="whitegrid")
X, y = load_data("../data")
summarize(X, y)

## Class balance

BRCA (breast) has far more samples than COAD (colon). That's why the training script reports macro F1 alongside accuracy: accuracy alone can hide poor performance on the smaller classes.

In [ ]:
ax = y.value_counts().plot.bar(color="#3182bd", figsize=(6, 3.5))
ax.set_ylabel("Samples")
ax.set_title("Samples per cancer type")
plt.xticks(rotation=0)
plt.show()

## Expression values

The values are already normalized and on a log-like scale, so no extra log transform is applied. Many genes are zero in most samples.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(X.values.ravel(), bins=100, color="#6baed6")
axes[0].set_title("All expression values")
axes[0].set_yscale("log")
axes[1].hist((X > 0).mean(), bins=50, color="#6baed6")
axes[1].set_title("Fraction of samples where each gene is expressed")
plt.tight_layout()
plt.show()

## Unsupervised views: PCA and t-SNE

No labels are used to compute these projections. Labels are only used for coloring.

In [ ]:
Xs = StandardScaler().fit_transform(X.loc[:, X.std() > 0])
pcs = PCA(n_components=50, random_state=0).fit_transform(Xs)
emb = TSNE(n_components=2, perplexity=30, random_state=0).fit_transform(pcs)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for label in sorted(y.unique()):
    m = (y == label).values
    axes[0].scatter(pcs[m, 0], pcs[m, 1], s=10, label=label)
    axes[1].scatter(emb[m, 0], emb[m, 1], s=10, label=label)
axes[0].set_title("PCA (PC1 vs PC2)")
axes[1].set_title("t-SNE on top 50 PCs")
axes[1].legend(title="Cancer type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Heatmap of the most variable genes

Hierarchical clustering on the 50 most variable genes. Samples from the same tissue tend to cluster together, which fits with tissue-of-origin being the strongest signal in tumor expression data.

In [ ]:
top_var = X.var().sort_values(ascending=False).head(50).index
palette = dict(zip(sorted(y.unique()), sns.color_palette("tab10")))
g = sns.clustermap(X[top_var].T, z_score=0, cmap="vlag", figsize=(10, 8),
                   col_colors=y.map(palette), xticklabels=False, yticklabels=False)
g.ax_heatmap.set_xlabel("Samples")
g.ax_heatmap.set_ylabel("Top 50 variable genes")
plt.show()

## Takeaways

The five tumor types separate clearly even without supervision, so near-perfect classification is expected. The more useful results from `src/train.py` are the ones that check for overfitting (cross-validation vs held-out test) and the list of genes the models rely on.